### hypird search--> combine dense and spares retriver

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import  BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader

In [32]:
textloadr=TextLoader('New Text Document.txt')
docs=textloadr.load()
docs[0]

Document(metadata={'source': 'New Text Document.txt'}, page_content='Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data. It includes techniques such as supervised learning, unsupervised learning, and reinforcement learning. Applications range from image recognition to natural language processing.\n\nClimate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities like burning fossil fuels. It leads to rising sea levels, extreme weather events, and loss of biodiversity. Governments and organizations are working on mitigation strategies and renewable energy adoption.\n\nThe history of space exploration began with the launch of Sputnik in 1957. Since then, humans have sent satellites, probes, and astronauts to explore space. Key milestones include landing on the Moon, establishing the International Space Station, and sending rovers to Mars.\n\nNutrition is the study of how food a

In [33]:
embedding=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 411.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
from langchain_experimental.text_splitter import SemanticChunker
chunker = SemanticChunker(
    embeddings=embedding,
    breakpoint_threshold_type="percentile",  # or "standard_deviation"
    breakpoint_threshold_amount=70           # lower = more splits
)
chunks=chunker.split_documents(docs)

In [47]:

dense_vector=FAISS.from_documents(chunks, embedding)
dense_retriever=dense_vector.as_retriever()


In [48]:
sparse_retriever=BM25Retriever.from_documents(chunks)
sparse_retriever.k=3


In [49]:
hypird_retriver=EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

In [50]:
query="What is the capital of France?"
results=hypird_retriver.invoke(query)
print(results)

[Document(id='3df4ecb1-86b6-4c3c-951d-f515b19614cc', metadata={'source': 'New Text Document.txt'}, page_content='Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data. It includes techniques such as supervised learning, unsupervised learning, and reinforcement learning.'), Document(id='a930a081-8a02-4bbf-9d82-23fd58c7ab6e', metadata={'source': 'New Text Document.txt'}, page_content='Governments and organizations are working on mitigation strategies and renewable energy adoption. The history of space exploration began with the launch of Sputnik in 1957. Since then, humans have sent satellites, probes, and astronauts to explore space.'), Document(id='7ed48ae0-5c38-4f90-b693-41be5f45e555', metadata={'source': 'New Text Document.txt'}, page_content='Key milestones include landing on the Moon, establishing the International Space Station, and sending rovers to Mars. Nutrition is the study of how food affects the health and growth of

In [51]:
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain.chat_models.base import init_chat_model
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
load_dotenv()
llm = init_chat_model("groq:llama-3.3-70b-versatile", temperature=0.7)


In [52]:
prompt=PromptTemplate.from_template(
    "Answer the question based on the following retrieved documents:\n\n{context}\n\nQuestion: {input}\nAnswer:"
)

In [55]:
chain_docs=create_stuff_documents_chain(llm=llm, prompt=prompt)
retrieval_chain=create_retrieval_chain(retriever=hypird_retriver, combine_docs_chain=chain_docs)
response = retrieval_chain.invoke({
    "input": "what is field of art?"
})
print(response["answer"])

Based on the provided documents, the field of art mentioned is photography. It is described as "a creative expression and a documentation tool" that captures moments using cameras and light, and is influenced by techniques such as composition, lighting, and post-processing.
